## Импорты

In [ ]:
import requests
import time
import csv
import os
from datetime import datetime

Парсинг вакансий HH.ru

In [ ]:
def get_vacancies(pages=1, records=50):
    url = "https://api.hh.ru/vacancies"
    vacancies = []

    for page in range(pages):
        print(f'Обработка страницы {page + 1}/{pages}')

        params = {
            "per_page": records,
            "page": page,
            "order_by": "publication_time"
        }

        headers = {'User-Agent': 'MyHHApp/1.0 (chunai6@mail.ru)'}
        response = requests.get(url, params=params, headers=headers)

        if response.status_code == 200:
            data = response.json()
            vacancies.extend(data['items'])
        else:
            print(f"Ошибка {response.status_code}")
            break

        time.sleep(1)

    return vacancies


Сохранение в CSV

In [ ]:
def save_vacancies_to_csv(data, filename):
    if not data:
        print("Нет данных для сохранения")
        return

    with open(filename, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow([
            'Название', 'ID', 'Дата парсинга', 'Регион', 'Работодатель',
            'Зарплата (н)', 'Зарплата(в)', 'Валюта', 'Рабочий день',
            'Рабочие часы', 'График', 'Тип занятости', 'Форма занятости',
            'Дата публикации', 'Архивное', 'Краткое описание',
            'Полное описание', 'URL'
        ])

        for v in data:
            name = v.get('name')
            vid = v.get('id')
            date_of_parsing = datetime.now()
            region = v.get('area', {}).get('name')
            employer = v.get('employer', {}).get('name')

            salary = v.get('salary')
            salary_from = salary.get('from') if salary else ""
            salary_to = salary.get('to') if salary else ""
            salary_currency = salary.get('currency') if salary else ""

            schedule = v.get('schedule', {}).get('name') if v.get('schedule') else ""

            working_hours = ""
            if v.get('working_hours') and len(v.get('working_hours')) > 0:
                working_hours = v.get('working_hours')[0].get('name')

            schedule_days = ""
            if v.get('work_schedule_by_days') and len(v.get('work_schedule_by_days')) > 0:
                schedule_days = v.get('work_schedule_by_days')[0].get('name')

            employment = v.get('employment', {}).get('name') if v.get('employment') else ""
            employment_form = v.get('employment_form', {}).get('name') if v.get('employment_form') else ""
            published_at = v.get('published_at')
            archived = v.get('archived')

            requirement = v.get('snippet', {}).get('requirement', '') if v.get('snippet') else ""
            responsibility = v.get('snippet', {}).get('responsibility', '') if v.get('snippet') else ""
            vurl = v.get('alternate_url')

            writer.writerow([
                name, vid, date_of_parsing, region, employer, salary_from,
                salary_to, salary_currency, schedule, working_hours, schedule_days,
                employment, employment_form, published_at, archived,
                requirement, responsibility, vurl
            ])

    print(f"Сохранено {len(data)} вакансий в {filename}")

Основной запуск

In [ ]:
if __name__ == "__main__":
    data = get_vacancies(pages=3, records=50)

    desktop = os.path.join(os.path.expanduser("~"), "Desktop")
    filename = os.path.join(desktop, "hh_vacancies.csv")

    save_vacancies_to_csv(data, filename)